# Solar TRACE Exploration

Exploratory analysis of SolarTRACE Dataset v9-9-2025.xlsx.

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append("../modules")

XLSX = "../data/SolarTRACE Dataset v9-9-2025.xlsx"


In [ ]:
ahj_timelines     = pd.read_excel(XLSX, sheet_name="AHJ-Utility Timelines")
state_timelines   = pd.read_excel(XLSX, sheet_name="State-Level Timelines")
ahj_permit_reqs   = pd.read_excel(XLSX, sheet_name="AHJ permits reqs")
utility_ix_reqs   = pd.read_excel(XLSX, sheet_name="Utility IX reqs")

print("AHJ timelines:", ahj_timelines.shape)
print("State timelines:", state_timelines.shape)
print("AHJ permit reqs:", ahj_permit_reqs.shape)
print("Utility IX reqs:", utility_ix_reqs.shape)


## Interconnection Timelines


### Year selection: 2023 vs 2024

The dataset includes data through 2024, but we first check whether 2024 is fully reported.


In [ ]:
size_techs = [
    '0-10kW PV Only',
    '10-20kW PV Only',
    '0-10kW PV+Storage',
    '10-20kW PV+Storage',
]

rows = []
for st in size_techs:
    col23 = f'Installs 2023 {st}'
    col24 = f'Installs 2024 {st}'
    tmp = state_timelines[['state', col23, col24]].copy()
    tmp.columns = ['state', 'installs_2023', 'installs_2024']
    tmp['size_tech'] = st
    rows.append(tmp)

yoy = pd.concat(rows, ignore_index=True)
yoy['installs_2023'] = pd.to_numeric(yoy['installs_2023'], errors='coerce')
yoy['installs_2024'] = pd.to_numeric(yoy['installs_2024'], errors='coerce')
yoy['change'] = yoy['installs_2024'] - yoy['installs_2023']
yoy['pct_change'] = (yoy['change'] / yoy['installs_2023']).round(3)

summary = yoy.groupby('size_tech')[['installs_2023', 'installs_2024', 'change']].sum()
summary['pct_change'] = (summary['change'] / summary['installs_2023']).round(3)
summary


0-10kW PV Only dropped 72% from 2023 to 2024 — far too large to be a real market decline, and the pattern is consistent across all size/tech classes. This strongly suggests incomplete 2024 reporting at the time of the September 2025 data pull. **We use 2023 as the reference year.**


### Size class selection: 0-10kW vs 10-20kW

Check 2023 install counts by state to determine which size class has better coverage.


In [ ]:
cols = {
    '0-10kW PV Only':     'Installs 2023 0-10kW PV Only',
    '10-20kW PV Only':    'Installs 2023 10-20kW PV Only',
    '0-10kW PV+Storage':  'Installs 2023 0-10kW PV+Storage',
    '10-20kW PV+Storage': 'Installs 2023 10-20kW PV+Storage',
}

comparison = state_timelines[['state'] + list(cols.values())].copy()
comparison.columns = ['state'] + list(cols.keys())

comparison['total_0-10kW']  = comparison['0-10kW PV Only'].fillna(0)  + comparison['0-10kW PV+Storage'].fillna(0)
comparison['total_10-20kW'] = comparison['10-20kW PV Only'].fillna(0) + comparison['10-20kW PV+Storage'].fillna(0)
comparison['pct_10_20'] = (
    comparison['total_10-20kW'] / (comparison['total_0-10kW'] + comparison['total_10-20kW'])
).round(3)

comparison.sort_values('total_0-10kW', ascending=False)


In [ ]:
# 2023: installs + pre-install IX + final IX to PTO, by state, for 0-10kW vs 10-20kW (PV Only)
ix_cols = {
    'installs_0_10':      'Installs 2023 0-10kW PV Only',
    'pre_ix_0_10':        'Median Pre-Install IX Time 2023 0-10kW PV Only',
    'final_ix_0_10':      'Median Final IX to PTO 2023 0-10kW PV Only',
    'installs_10_20':     'Installs 2023 10-20kW PV Only',
    'pre_ix_10_20':       'Median Pre-Install IX Time 2023 10-20kW PV Only',
    'final_ix_10_20':     'Median Final IX to PTO 2023 10-20kW PV Only',
}

ix = state_timelines[['state'] + list(ix_cols.values())].copy()
ix.columns = ['state'] + list(ix_cols.keys())

for col in ix_cols.keys():
    ix[col] = pd.to_numeric(ix[col], errors='coerce')

# Differences between size classes
ix['pre_ix_diff']   = (ix['pre_ix_10_20']   - ix['pre_ix_0_10']).round(1)
ix['final_ix_diff'] = (ix['final_ix_10_20'] - ix['final_ix_0_10']).round(1)

ix.sort_values('installs_0_10', ascending=False)


## QA/QC: utility_timelines & state_timelines

In [ ]:
# Solar TRACE timelines — QA/QC

SOLARTRACE_PATH = "../data/SolarTRACE Dataset v9-9-2025.xlsx"

utility_timelines, state_timelines = solartrace_run_pipeline(SOLARTRACE_PATH)

# Utility spot-check: Eversource and PSE&G, 2023
print("=== Eversource Energy (NSTAR) & PSE&G — 2023 ===")
_util_check = utility_timelines[
    (utility_timelines["year"] == 2023) &
    (utility_timelines["utility"].isin([
        "Eversource Energy (NSTAR)",
        "Public Service Elec & Gas Co (PSE&G)",
    ]))
]
display(_util_check)

# State spot-check: CA 2023, 0-10kW PV Only
print("\n=== state_timelines — CA 2023 0_10kw pv_only ===")
_state_check = state_timelines[
    (state_timelines["state"] == "CA") &
    (state_timelines["year"] == 2023) &
    (state_timelines["size_class"] == "0_10kw") &
    (state_timelines["tech_class"] == "pv_only")
]
display(_state_check)


## Comparison: Solar TRACE vs old static CSVs
New: Solar TRACE 2023, 0-10kW, PV Only. Old: `state_inspection_timelines_pv.csv` (inspection) and `state_median_permitting_timelines_by_tech.csv` (OHM 2024, solar).

In [21]:
from process_solartrace_timelines import run_pipeline as solartrace_run_pipeline

_STATE_ABBR = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA',
    'Colorado':'CO','Connecticut':'CT','Delaware':'DE','District of Columbia':'DC',
    'Florida':'FL','Georgia':'GA','Hawaii':'HI','Idaho':'ID','Illinois':'IL',
    'Indiana':'IN','Iowa':'IA','Kansas':'KS','Kentucky':'KY','Louisiana':'LA',
    'Maine':'ME','Maryland':'MD','Massachusetts':'MA','Michigan':'MI','Minnesota':'MN',
    'Mississippi':'MS','Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV',
    'New Hampshire':'NH','New Jersey':'NJ','New Mexico':'NM','New York':'NY',
    'North Carolina':'NC','North Dakota':'ND','Ohio':'OH','Oklahoma':'OK','Oregon':'OR',
    'Pennsylvania':'PA','Puerto Rico':'PR','Rhode Island':'RI','South Carolina':'SC',
    'South Dakota':'SD','Tennessee':'TN','Texas':'TX','Utah':'UT','Vermont':'VT',
    'Virginia':'VA','Washington':'WA','West Virginia':'WV','Wisconsin':'WI','Wyoming':'WY',
}

_, _state_df = solartrace_run_pipeline(XLSX)

_new = _state_df[
    (_state_df['year'] == 2023) &
    (_state_df['size_class'] == '0_10kw') &
    (_state_df['tech_class'] == 'pv_only')
][['state', 'installs', 'permit_time_days', 'inspection_time_days']].copy()
_new.columns = ['state', 'installs', 'permit_new', 'inspection_new']

_insp = pd.read_csv('../data/state_inspection_timelines_pv.csv')[['state', 'median_inspection_timeline_days']]
_insp['state'] = _insp['state'].map(_STATE_ABBR)
_insp.columns = ['state', 'inspection_old']

_perm = pd.read_csv('../data/state_median_permitting_timelines_by_tech.csv')
_perm = _perm[(_perm['technology'] == 'solar') & (_perm['year'] == 2024)][['state', 'median_permitting_timeline_days']]
_perm['state'] = _perm['state'].map(_STATE_ABBR)
_perm.columns = ['state', 'permit_old']

_comp = _new.merge(_insp, on='state', how='outer').merge(_perm, on='state', how='outer')
_comp['permit_diff'] = _comp['permit_new'] - _comp['permit_old']
_comp['inspection_diff'] = _comp['inspection_new'] - _comp['inspection_old']
_comp = _comp[['state', 'installs', 'permit_old', 'permit_new', 'permit_diff', 'inspection_old', 'inspection_new', 'inspection_diff']].sort_values('installs', ascending=False).reset_index(drop=True)

# Difference distribution
_bins   = [-999, -7, -4, -1, 0, 1, 4, 7, 999]
_labels = ['≤ -7', '-4 to -6', '-1 to -3', '0', '+1 to +3', '+4 to +6', '+7 to +9', '≥ +10']

_rows = []
for metric, col in [('Permit', 'permit_diff'), ('Inspection', 'inspection_diff')]:
    d = _comp[col].dropna()
    counts = pd.cut(d, bins=_bins, labels=_labels).value_counts().reindex(_labels).fillna(0).astype(int)
    counts.name = metric
    _rows.append(counts)

_summary = pd.DataFrame(_rows)
_summary.index.name = 'metric'
print(f"Difference distribution (new minus old), n states with both sources:")
display(_summary)
print()
display(_comp)


Difference distribution (new minus old), n states with both sources:


,≤ -7,-4 to -6,-1 to -3,0,+1 to +3,+4 to +6,+7 to +9,≥ +10
metric,,,,,,,,
Permit,2,3,11,2,0,9,0,6
Inspection,2,3,5,7,4,7,1,1


,state,installs,permit_old,permit_new,permit_diff,inspection_old,inspection_new,inspection_diff
0,CA,40259.0,5.0,4.0,-1.0,12.50,11.0,-1.50
1,TX,5066.0,3.0,0.0,-3.0,9.00,5.0,-4.00
2,AZ,4875.0,16.0,4.0,-12.0,11.00,11.0,0.00
3,CO,3820.0,3.0,2.0,-1.0,15.00,18.0,3.00
4,NV,3631.0,4.0,0.0,-4.0,12.00,9.0,-3.00
5,FL,3605.0,7.0,7.0,0.0,13.00,13.0,0.00
6,IL,3136.0,12.0,10.0,-2.0,10.00,14.0,4.00
7,MA,2933.0,3.0,6.0,3.0,12.00,11.0,-1.00
8,NJ,1945.0,12.0,10.5,-1.5,19.00,17.0,-2.00
9,NM,1545.0,7.0,6.0,-1.0,15.75,16.0,0.25
